# Visual Analytics

## Assignment 3

**Instructor:** Dr. Marco D'Ambros  
**TAs:** Giuseppe Crupi, Mattia Giannaccari

**Contacts:** marco.dambros@usi.ch, giuseppe.crupi@usi.ch, mattia.giannaccari@usi.ch

**Due Date:** May 25, 2026 @ 23:55

---
The goal of this assignment is to use **Spark (PySpark)** and **Polars** in Jupyter notebooks.  
The files `trip_data.csv`, `trip_fare.csv`, and `nyc_boroughs.geojson` are available in the provided folder: [Assignment3-data](https://usi365-my.sharepoint.com/:f:/g/personal/armenc_usi_ch/Ejp7sb8QAMROoWe0XUDcAkMBoqUFk-w2Vgroup025NhAww?e=2I7SMC).

- Use **Spark** to solve **Exercises 1–4**
- Use **Polars** to solve **Exercises 5–8**

Please name your notebook file as `SurnameName_Assignment3.ipynb`

# ⚡️ Spark Exercises (50 pts)

### Importing the libraries

In [1]:
import os
import json

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

from shapely.geometry import shape, Point

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, FactorRange, Legend, LegendItem
from bokeh.transform import factor_cmap
from bokeh.palettes import Spectral4

### Initial Spark Setup

In [2]:
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = "python"

spark = SparkSession.builder \
    .appName("Assignment3") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 12:40:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



### Importing and cleaning the data

In [3]:
# Loading the datasets
# After selecting the dataset thanks to the related paths, header=True is used to consider the first row as column names while 
# inferSchema=True is used to automatically detect data types

trip_data = spark.read.csv(
    "data-assignment3/trip_data.csv", header=True, inferSchema=True
)

trip_fare = spark.read.csv(
    "data-assignment3/trip_fare.csv", header=True, inferSchema=True
)

# Since the column names have spaces, they need to be trimmed before joining
trip_data = trip_data.toDF(*[c.strip() for c in trip_data.columns])
trip_fare = trip_fare.toDF(*[c.strip() for c in trip_fare.columns])

# Visualizing the clean datasets without spaces in column names
trip_data.show(5)
trip_fare.show(5)

+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+
|           medallion|        hack_license|vendor_id|rate_code|store_and_fwd_flag|    pickup_datetime|   dropoff_datetime|passenger_count|trip_time_in_secs|trip_distance|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|
+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+
|89D227B655E5C82AE...|BA96DE419E711691B...|      CMT|        1|                 N|2013-01-01 15:11:48|2013-01-01 15:18:10|              4|              382|          1.0|      -73.978165|      40.757977|       -73.989838|       40.751171|
|0BD7C8F5BA12B88E0...|9FD8F69F0804BDB55...| 

### Exercise 1 (8 pts)
Join the `trip_data` and `trip_fare` dataframes into one, considering only trips from January 1–7, 2013. Filter out trips where the total amount charged is 0 or less, or the trip distance is 0 or less. Report how many rows are removed by each filter, the total number of rows after filtering, and the average tip amount across the resulting dataset.

### Solution 1

#### Merging the two dataframes

In [4]:
# Merging the datasets on the selected columns: medallion, hack_license, pickup_datetime
# This is because the medallion and hack_license columns identify the taxi and the driver, 
# while the pickup_datetime column identifies the specific trip.
df = trip_data.join(
    trip_fare,
    on=["medallion", "hack_license", "pickup_datetime"],
    how="inner"
)

print(f"Rows after merging: {df.count()}")

Rows after merging: 14776615


#### Filtering trips and showing results

In [5]:
# PICKUP DATETIME FILTER
rows_before_date = df.count()

# Filter trips between January 1 and January 7, 2013
df = df.filter(
    (F.col("pickup_datetime") >= "2013-01-01") &
    (F.col("pickup_datetime") <  "2013-01-08")
)

rows_after_date = df.count()
print(f"Rows removed after date filtering: {rows_before_date - rows_after_date}")


# TOTAL AMOUNT FILTER
rows_before_amount = df.count()

# Filter trips with total_amount > 0, the ones with total_amount equal to or less than 0 
# are likely to be wrong.
df = df.filter(F.col("total_amount") > 0)

rows_after_amount = df.count()
print(f"Rows removed by total_amount filter: {rows_before_amount - rows_after_amount}")


# TRIP DISTANCE FILTER
rows_before_dist = df.count()

# Filter trips with trip_distance > 0, the ones with trip_distance equal to or less than 0
# are likely to be wrong.
df = df.filter(F.col("trip_distance") > 0)

rows_after_dist = df.count()
print(f"Rows removed by trip_distance filter: {rows_before_dist - rows_after_dist}")


# FINAL RESULTS
avg_tip = df.agg(F.avg("tip_amount")).collect()[0][0]

print(f"Total rows after all filters: {df.count()}")
print(f"Average tip amount: ${avg_tip:.2f}")

Rows removed after date filtering: 11766480


Rows removed by total_amount filter: 0


Rows removed by trip_distance filter: 19645


Total rows after all filters: 2990490
Average tip amount: $1.13


### Exercise 2 (12 pts)
For each hour of the day (0–23), compute the average trip duration in minutes and the average trip distance. Provide a graphical representation that allows comparing both metrics across hours side by side. You may want to have a look at: https://docs.bokeh.org/en/latest/docs/user_guide/basic/bars.html#grouping

### Solution 2

In [6]:
# Cell 1 - Compute avg trip duration (minutes) and avg distance per hour
hourly_stats = df.groupBy(F.hour("pickup_datetime").alias("hour")) \
    .agg(
        F.avg(F.col("trip_time_in_secs") / 60).alias("avg_duration_min"),
        F.avg("trip_distance").alias("avg_distance")
    ) \
    .orderBy("hour")

hourly_stats.show(24)

+----+------------------+------------------+
|hour|  avg_duration_min|      avg_distance|
+----+------------------+------------------+
|   0|11.223678905155799|3.1824580652573524|
|   1|11.414620295666214| 3.187834960243697|
|   2|11.304379693818616| 3.266443770460237|
|   3|11.261855373762803| 3.523590795382152|
|   4|11.762098527361886| 4.130701579528958|
|   5|12.035606561144466| 4.981900548677145|
|   6|10.405564935503474| 3.765743567231717|
|   7|10.550317880700412| 3.146331414658111|
|   8|11.212508254337276|  2.77326200494442|
|   9|10.773916100997631|2.6097442147485355|
|  10|10.651816069479327| 2.713621435157219|
|  11|10.621733516892398| 2.640079624944189|
|  12|10.682610332471548|2.6117908558987577|
|  13| 11.18953992443917|2.7550136559767515|
|  14|11.869509862600193|2.9462623607785976|
|  15|11.826220664083433|2.9449250072488575|
|  16| 11.36870395998212|2.8806185039426806|
|  17|11.609118601008788|2.7296197294264157|
|  18|11.264193100947535| 2.587695134895363|
|  19|10.5

In [7]:
output_notebook()

stats = hourly_stats.collect()
hours = [str(row["hour"]) for row in stats]
avg_duration = [row["avg_duration_min"] for row in stats]
avg_distance = [row["avg_distance"] for row in stats]

x = [(h, metric) for h in hours for metric in ["Duration (min)", "Distance (miles)"]]
counts = []
for i in range(len(hours)):
    counts.append(avg_duration[i])
    counts.append(avg_distance[i])

source = ColumnDataSource(dict(x=x, counts=counts))

p = figure(
    x_range=FactorRange(*x),
    height=500,
    width=1200,
    title="Average Trip Duration and Distance by Hour of Day (Jan 1–7, 2013)",
    toolbar_location=None
)

# Make title bigger and bold
p.title.text_font_size = "16px"
p.title.text_font_style = "bold"
p.title.align = "center"

palette = ["steelblue", "orange"]
factors = ["Duration (min)", "Distance (miles)"]

r = p.vbar(
    x="x",
    top="counts",
    width=0.9,
    source=source,
    fill_color=factor_cmap("x", palette=palette, factors=factors, start=1, end=2)
)

# Legend top-right, outside the bars
legend = Legend(items=[
    LegendItem(label="Duration (min)",   renderers=[r], index=0),
    LegendItem(label="Distance (miles)", renderers=[r], index=1),
], location="top_right")


# Move legend outside the plot area (top)
p.add_layout(legend)

# Useful for not overlapping the legend with the bars
p.y_range.start = 0
p.y_range.end   = max(avg_duration) * 1.15

# Remove x-axis tick labels (no more rotated text under bars)
p.xaxis.major_label_text_font_size = "0pt"
p.xgrid.grid_line_color = None
p.yaxis.axis_label = "Value"
p.xaxis.axis_label = "Hour of the Day"
p.xaxis.major_tick_line_color = None  # remove individual bar ticks
p.xaxis.minor_tick_line_color = None  # remove minor ticks
p.xaxis.group_label_orientation = 0
p.xaxis.group_text_font_size = "11px"

show(p)

Loading BokehJS ...

### Exercise 3 (14 pts)
Consider only the boroughs Queens, Staten Island, and EWR. Create a dataframe that shows, for each payment type, the total fare amount collected for trips *originating from* each of those three boroughs, broken down by *destination borough* (including all boroughs as destinations).

> For example, for Queens you should consider:
> - Queens → Queens (cash), Queens → Queens (card), ...
> - Queens → Manhattan (cash), Queens → Manhattan (card), ...
> - and so on for all destination boroughs.


### Solution 3

#### Loading the NYC Boroughs GeoJSON

In [8]:
# Load GeoJSON
with open("data-assignment3/nyc-boroughs.geojson", "r") as f:
    geojson = json.load(f)

borough_shapes = [
    (feature['properties']['borough'], shape(feature['geometry']))
    for feature in geojson['features']
]

# Broadcast to all workers
shapes_broadcast = spark.sparkContext.broadcast(borough_shapes)

#### Defining a function for retrieving the borough based on the coordinates

In [9]:
# Function to determine the borough based on longitude and latitude
def get_borough(lon, lat):
    if lon is None or lat is None:
        return None
    point = Point(lon, lat)
    for name, geom in shapes_broadcast.value:
        if geom.contains(point):
            return name
    return "EWR"# points outside all polygons

# Custom operation to get the borough from longitude and latitude
get_borough_udf = F.udf(get_borough, StringType())

#### Adding names of the boroughs to the original dataframe

In [10]:
# Updating the original DataFrame with pickup and dropoff boroughs
df_boroughs = df.withColumn(
    "pickup_borough",
    get_borough_udf(F.col("pickup_longitude"), F.col("pickup_latitude"))
).withColumn(
    "dropoff_borough",
    get_borough_udf(F.col("dropoff_longitude"), F.col("dropoff_latitude"))
).cache()

#### Collecting results for trips with pickups in specific boroughs

In [ ]:
# List of boroughs to keep as pickup locations
selected_boroughs = ['Queens', 'Staten Island', 'EWR']

# Dataframe with total fare amount by pickup_borough, dropoff_borough, and payment_type for selected pickup boroughs
df_result = df_boroughs.filter(F.col("pickup_borough").isin(selected_boroughs)) \
    .groupBy("pickup_borough", "dropoff_borough", "payment_type") \
    .agg(F.round(F.sum("fare_amount"), 2).alias("total_fare_amount")) \
    .orderBy("pickup_borough", "dropoff_borough", "payment_type")

df_result.show(truncate=False)

26/05/14 12:44:45 WARN MemoryStore: Not enough space to cache rdd_255_146 in memory! (computed 1751.3 KiB so far)
26/05/14 12:44:45 WARN BlockManager: Persisting block rdd_255_146 to disk instead.
26/05/14 12:44:45 WARN MemoryStore: Not enough space to cache rdd_255_146 in memory! (computed 1751.3 KiB so far)
26/05/14 12:44:45 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_255_158 in memory.
26/05/14 12:44:49 WARN MemoryStore: Not enough space to cache rdd_255_158 in memory! (computed 504.0 B so far)
26/05/14 12:44:49 WARN BlockManager: Persisting block rdd_255_158 to disk instead.
26/05/14 12:44:51 WARN MemoryStore: Not enough space to cache rdd_255_156 in memory! (computed 1749.5 KiB so far)
26/05/14 12:44:51 WARN BlockManager: Persisting block rdd_255_156 to disk instead.
26/05/14 12:44:51 WARN MemoryStore: Not enough space to cache rdd_255_159 in memory! (computed 1751.2 KiB so far)
26/05/14 12:44:51 WARN BlockManager: Persisting 

+--------------+---------------+------------+-----------------+
|pickup_borough|dropoff_borough|payment_type|total_fare_amount|
+--------------+---------------+------------+-----------------+
|EWR           |Bronx          |CRD         |676.0            |
|EWR           |Bronx          |CSH         |1022.5           |
|EWR           |Bronx          |NOC         |17.0             |
|EWR           |Bronx          |UNK         |37.5             |
|EWR           |Brooklyn       |CRD         |3643.0           |
|EWR           |Brooklyn       |CSH         |2733.5           |
|EWR           |Brooklyn       |UNK         |299.5            |
|EWR           |EWR            |CRD         |325301.82        |
|EWR           |EWR            |CSH         |290788.29        |
|EWR           |EWR            |DIS         |589.5            |
|EWR           |EWR            |NOC         |2093.0           |
|EWR           |EWR            |UNK         |1111.0           |
|EWR           |Manhattan      |CRD     

### Exercise 4 (16 pts)
Create a dataframe where each row represents a driver, and there is one column per hour of the day (0–23). For each driver-hour, the dataframe provides the maximum number of consecutive trips where the tip amount was strictly greater than $0.

> For example, if for driver B we have trips starting in hour 14 (sorted by pickup time):
>
> - Trip 1: tip = $2.00
> - Trip 2: tip = $0.00
> - Trip 3: tip = $1.50
> - Trip 4: tip = $3.00
>
> The longest streak of tipped trips in hour 14 is 2 (Trips 3 and 4).

Additionally, print the pair (driver, hour) with the maximum streak.

### Solution 4

# 🐻‍❄️ Polars Exercises (50 pts)

In this section, you will use **Polars** to perform data cleaning, transformation, and analysis on the NYC taxi dataset.

You will work with the merged dataset obtained from:
- `trip_data.csv`
- `trip_fare.csv`

### Exercise 5 (10 pts)

Perform a sequence of data cleaning steps on the dataset:

1. Remove trips where:
   - `trip_distance <= 10` but `fare_amount > 100`.
   - `trip_distance > 100` or `trip_distance <= 0>` miles.

2. Remove trips with:
   - missing timestamps (`pickup_datetime`, `dropoff_datetime`).
   - `dropoff_datetime <= pickup_datetime`.

After each step:
- Report how many rows were removed.

Finally:
- Report the number of remaining rows.
- Check whether duplicate records exist (based on `medallion`, `hack_license`, `pickup_datetime`).

### Solution 5

### Exercise 6 (12 pts)

Analyze temporal patterns in taxi demand:

1. Group the data by `(weekday, hour)` and compute:
   - total number of trips
   - average fare per trip

2. Visualize the results using a **heatmap**

3. Return the top 5 `(weekday, hour)` by average fare.

### Solution 6

### Exercise 7 (12 pts)

Define a *high-value trip* as one satisfying **at least two** of the following conditions:

- `fare_amount` is in the top 10%
- `tip_amount > 50%` of `fare_amount`
- `trip_distance < 2 miles` AND `fare_amount` above the median

Tasks:

1. Extract all high-value trips.
2. Select only the rides longer than 10 miles (in a straight line).
3. Report the total number of such trips.
4. Create a scatterplot:
   - x-axis: `trip_distance`
   - y-axis: `fare_amount`

4. Briefly interpret the observed patterns

### Solution 7

### Exercise 8 (16 pts)

Analyze driver performance using earnings efficiency:

1. For each trip, compute:
   - trip duration in hours.
   - total earnings = `fare_amount + tip_amount`.

2. Filter:
   - only keep durations between `3 minutes and 5 hours`.

3. For each driver (`hack_license`), compute:
   - total earnings.
   - total driving time (in hours).
   - earnings per hour.
   - total number of trips.

4. Select the **top 15% drivers** based on number of trips.

5. Classify trips into:
   - **day** (i.e., `06:00–18:00`).
   - **night** (remaining hours).

6. Compare driver efficiency:
   - Plot the distribution of earnings per hour for `day vs night drivers` (notice that a driver can be both a "day" and "night" driver in case it performed at least one day ride and one night ride).

7. Answer:
   - Which group appears more efficient?
   - Provide a short explanation based on your results.

### Solution 8